# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rowan-ali/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline Objective

This notebook builds a transparent, rule-based action baseline for ranking client-content observations for review.

The workflow is intentionally evidence-first:

1. Audit two candidate signals using visible bucket-level evidence and sample counts.
2. Assign a simple verdict to each signal before using it in the rule.
3. Encode one fixed rule with a transparent score, one reason code, and one action label.
4. Rank the observations and review the highest-ranked items skeptically.
5. Check explicitly for future-window information, label-derived inputs, and other leakage.

The baseline is designed as decision support, not as a claim about Google's ranking algorithm.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"

MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("FlyRank warehouse connection: ready")
print("Analysis slice: March 2026")

FlyRank warehouse connection: ready
Analysis slice: March 2026


### Decision-Moment Data

The analysis uses the March 2026 daily performance slice established in the Week 3 data contract. The observation grain is the client-content-day level.

Only information available in the selected decision window is used for the baseline. The ranking proxy `gsc_clicks` is kept separate from the rule inputs to avoid direct label leakage.

The rule will rely on observable search and engagement signals and will not use future-window outcomes.

## Signal Audit — Evidence Before Rule Design

Before encoding the baseline rule, I first inspect the available fields and test candidate signals independently.

The goal is not to assume that a familiar SEO signal is useful. Each candidate must show a clear and interpretable pattern in the observed data before it is allowed to influence the baseline.

The audit will prioritize signals that are available at the decision moment and will exclude future-window information and label-derived inputs.

In [2]:
fact_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM {MAR}
    LIMIT 0
    """
).df()

print("Fact table columns:")
fact_schema[["column_name", "column_type"]].to_string(index=False)

Fact table columns:


'             column_name column_type\n             report_date        DATE\n          client_hash_id     VARCHAR\n         content_hash_id     VARCHAR\n          client_has_gsc     BOOLEAN\n          client_has_ga4     BOOLEAN\n      gsc_data_available     BOOLEAN\n      ga4_data_available     BOOLEAN\n         gsc_impressions      BIGINT\n              gsc_clicks      BIGINT\n        gsc_sum_position      BIGINT\n        gsc_avg_position      DOUBLE\n           ga4_pageviews      BIGINT\n            ga4_sessions      BIGINT\n               ga4_users      BIGINT\n    ga4_engaged_sessions      BIGINT\nga4_total_engagement_sec      BIGINT\n        sessions_organic      BIGINT\n         sessions_direct      BIGINT\n       sessions_referral      BIGINT\n         sessions_social      BIGINT\n           sessions_paid      BIGINT\n             sessions_ai      BIGINT\n              ai_chatgpt      BIGINT\n           ai_perplexity      BIGINT\n               ai_gemini      BIGINT\n           

In [3]:
content_schema = con.execute(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{REL}/dim_content.parquet')
    LIMIT 0
    """
).df()

print("Content dimension columns:")
print(content_schema[["column_name", "column_type"]].to_string(index=False))

Content dimension columns:
               column_name column_type
            client_hash_id     VARCHAR
           content_hash_id     VARCHAR
           keyword_hash_id     VARCHAR
               url_hash_id     VARCHAR
        keyword_char_count      BIGINT
       keyword_token_count      BIGINT
            url_char_count      BIGINT
      content_created_date        DATE
      content_updated_date        DATE
              content_type     VARCHAR
             search_volume      BIGINT
               competition      DOUBLE
         competition_level     VARCHAR
                       cpc      DOUBLE
               main_intent     VARCHAR
                 backlinks      BIGINT
            category_count      BIGINT
      keyword_created_date        DATE
             provider_used     VARCHAR
                model_used     VARCHAR
                char_count      BIGINT
                word_count      BIGINT
       last_optimized_date        DATE
optimization_eligible_date        DAT

In [4]:
candidate_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_data_available",
    "gsc_data_available"
]

available_fact_columns = set(fact_schema["column_name"])

print("Candidate signal availability:")
for column in candidate_columns:
    status = "AVAILABLE" if column in available_fact_columns else "NOT FOUND"
    print(f"{column:28} {status}")

Candidate signal availability:
report_date                  AVAILABLE
client_hash_id               AVAILABLE
content_hash_id              AVAILABLE
gsc_impressions              AVAILABLE
gsc_avg_position             AVAILABLE
ga4_sessions                 AVAILABLE
ga4_users                    AVAILABLE
ga4_engaged_sessions         AVAILABLE
ga4_data_available           AVAILABLE
gsc_data_available           AVAILABLE


### Signal Check 1 — Optimization Staleness

**FlyRank flag linkage:** Refresh / staleness logic

Optimization staleness is tested as the number of days since the content was last optimized, measured at the March 2026 decision date.

This signal is relevant to the refresh-style flag because recently optimized content should generally require less immediate refresh attention, while older optimization history may indicate a stronger opportunity for review.

The signal is audited before it is used in the baseline rule. The bucket analysis below reports the observed distribution and sample count (`n`) for each staleness band.

In [5]:
staleness_audit = con.execute(
    f"""
    WITH base AS (
        SELECT
            f.report_date,
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_impressions,
            f.gsc_avg_position,
            c.last_optimized_date,
            CASE
                WHEN c.last_optimized_date IS NULL THEN 'Never optimized'
                WHEN date_diff('day', c.last_optimized_date, f.report_date) <= 30 THEN '0-30 days'
                WHEN date_diff('day', c.last_optimized_date, f.report_date) <= 90 THEN '31-90 days'
                WHEN date_diff('day', c.last_optimized_date, f.report_date) <= 180 THEN '91-180 days'
                ELSE '181+ days'
            END AS staleness_bucket
        FROM {MAR} AS f
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') AS c
            ON f.client_hash_id = c.client_hash_id
            AND f.content_hash_id = c.content_hash_id
    )
    SELECT
        staleness_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position
    FROM base
    GROUP BY staleness_bucket
    ORDER BY
        CASE staleness_bucket
            WHEN '0-30 days' THEN 1
            WHEN '31-90 days' THEN 2
            WHEN '91-180 days' THEN 3
            WHEN '181+ days' THEN 4
            WHEN 'Never optimized' THEN 5
        END
    """
).df()

staleness_audit

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,avg_impressions,avg_position
0,0-30 days,1286535,146.3,11.26
1,Never optimized,8554843,10.8,17.90


### Staleness Verdict

**Verdict:** PENDING DATA REVIEW

The verdict will be assigned after inspecting the bucket-level sample sizes and the direction of the observed performance pattern. The signal will only enter the baseline rule if the evidence supports a stable and interpretable relationship.

In [6]:
staleness_data_quality = con.execute(
    f"""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS n_with_impressions,
        COUNT(*) FILTER (WHERE gsc_avg_position > 0) AS n_with_position,
        COUNT(*) FILTER (WHERE gsc_data_available) AS n_gsc_available
    FROM {MAR}
    """
).df()

staleness_data_quality

,n_total,n_with_impressions,n_with_position,n_gsc_available
0,9841378,3611061,3447872,3611061


### Staleness Verdict

**Verdict: CONFIRMED**

The staleness signal shows a clear and interpretable relationship with observed search visibility. Recently optimized content (0–30 days) has substantially higher average impressions (146.3) and a better average search position (11.26) than content that has never been optimized (10.8 impressions and position 17.90).

The evidence supports the refresh-related hypothesis at the signal level. This does not imply causation: recently optimized content may differ from never-optimized content in other ways. The signal is therefore treated as a prioritization input rather than proof that optimization itself caused the observed performance difference.

The sample sizes are also materially different, with 1,286,535 recently optimized observations versus 8,554,843 never-optimized observations. This imbalance is reported explicitly rather than ignored.

GSC availability is limited to 3,611,061 of 9,841,378 observations, so the visibility comparison is interpreted within the available GSC observations rather than as a universal property of the entire dataset.

### Signal Check #2 — Search Volume

**FlyRank flag linkage:** Quick-win / volume prioritization logic.

Search volume is audited as a candidate prioritization signal because higher-demand queries can represent larger potential opportunity.

The hypothesis is directional: content associated with higher search volume should show stronger observed visibility or opportunity than content associated with very low search volume.

The signal is evaluated using the content-level `search_volume` field and March 2026 observations. No future outcome or label-derived field is used.

In [7]:
volume_audit = con.execute(
    f"""
    WITH base AS (
        SELECT
            f.content_hash_id,
            f.gsc_impressions,
            f.gsc_avg_position,
            c.search_volume
        FROM {MAR} f
        INNER JOIN read_parquet('{REL}/dim_content.parquet') c
            ON f.content_hash_id = c.content_hash_id
        WHERE c.is_deleted = FALSE
          AND c.search_volume IS NOT NULL
          AND c.search_volume >= 0
    ),
    bucketed AS (
        SELECT
            CASE
                WHEN search_volume = 0 THEN 'Zero'
                WHEN search_volume <= 100 THEN '1-100'
                WHEN search_volume <= 1000 THEN '101-1,000'
                WHEN search_volume <= 10000 THEN '1,001-10,000'
                ELSE '10,000+'
            END AS volume_bucket,
            gsc_impressions,
            gsc_avg_position
        FROM base
    )
    SELECT
        volume_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_impressions), 2) AS avg_impressions,
        ROUND(AVG(NULLIF(gsc_avg_position, 0)), 2) AS avg_position
    FROM bucketed
    GROUP BY 1
    ORDER BY
        CASE volume_bucket
            WHEN 'Zero' THEN 1
            WHEN '1-100' THEN 2
            WHEN '101-1,000' THEN 3
            WHEN '1,001-10,000' THEN 4
            WHEN '10,000+' THEN 5
        END
    """
).df()

volume_audit

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,volume_bucket,n,avg_impressions,avg_position
0,Zero,3740470,30.79,16.08
1,1-100,3535092,37.85,16.08
2,"101-1,000",633366,40.01,20.65
3,"1,001-10,000",125430,28.97,26.18
4,"10,000+",14558,45.92,24.70




### Search Volume Verdict

**Verdict: PENDING DATA REVIEW**

The verdict will be assigned after reviewing the bucket-level sample sizes and the direction of the observed relationship between search volume, impressions, and search position.



In [8]:
volume_quality = con.execute(
    f"""
    SELECT
        COUNT(*) AS n_total,
        COUNT(*) FILTER (WHERE c.search_volume IS NOT NULL) AS n_with_search_volume,
        COUNT(*) FILTER (WHERE c.search_volume > 0) AS n_positive_search_volume
    FROM {MAR} f
    INNER JOIN read_parquet('{REL}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id
    WHERE c.is_deleted = FALSE
    """
).df()

volume_quality

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_total,n_with_search_volume,n_positive_search_volume
0,9640288,8048916,4308446


### Search Volume Verdict

**Verdict: MIXED**

The search-volume signal does not show a stable monotonic relationship with observed search visibility or ranking position.

Average impressions increase from 30.79 for zero-volume content to 37.85 for the 1–100 bucket and 40.01 for the 101–1,000 bucket, then fall to 28.97 for the 1,001–10,000 bucket before rising to 45.92 for the 10,000+ bucket. Average position also becomes worse in the higher-volume groups, moving from 16.08 in the zero and 1–100 buckets to 20.65, 26.18, and 24.70 in the higher-volume buckets.

The 10,000+ group is particularly small (n = 14,558), so its higher average impressions should not be treated as strong standalone evidence.

Search volume is therefore retained as contextual information rather than used as a direct scoring signal in the baseline rule. This conservative decision avoids turning a noisy and non-monotonic relationship into an unjustified prioritization rule.

### Final Staleness Robustness Check

The initial staleness comparison showed a strong separation in observed impressions and average position. Before using staleness in the baseline score, I check whether the relationship is being driven by differences in GSC availability across the two groups.

This check reports GSC coverage together with search visibility metrics for each staleness bucket. The purpose is to make the final rule conservative and avoid treating missing search data as evidence of low performance.

In [9]:
staleness_robustness = con.execute(
    f"""
    WITH base AS (
        SELECT
            f.gsc_data_available,
            f.gsc_impressions,
            f.gsc_avg_position,
            c.last_optimized_date
        FROM {MAR} f
        INNER JOIN read_parquet('{REL}/dim_content.parquet') c
            ON f.content_hash_id = c.content_hash_id
        WHERE c.is_deleted = FALSE
    ),
    bucketed AS (
        SELECT
            CASE
                WHEN last_optimized_date IS NULL THEN 'Never optimized'
                WHEN DATE '2026-03-31' - last_optimized_date < 30 THEN '0-30 days'
                ELSE '30+ days'
            END AS staleness_bucket,
            gsc_data_available,
            gsc_impressions,
            gsc_avg_position
        FROM base
    )
    SELECT
        staleness_bucket,
        COUNT(*) AS n,
        COUNT(*) FILTER (WHERE gsc_data_available) AS n_gsc_available,
        ROUND(
            100.0 * COUNT(*) FILTER (WHERE gsc_data_available) / COUNT(*),
            2
        ) AS gsc_available_pct,
        ROUND(
            AVG(gsc_impressions) FILTER (WHERE gsc_data_available),
            2
        ) AS avg_impressions_when_available,
        ROUND(
            AVG(NULLIF(gsc_avg_position, 0))
            FILTER (WHERE gsc_data_available),
            2
        ) AS avg_position_when_available
    FROM bucketed
    GROUP BY 1
    ORDER BY
        CASE staleness_bucket
            WHEN '0-30 days' THEN 1
            WHEN '30+ days' THEN 2
            ELSE 3
        END
    """
).df()

staleness_robustness

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,n_gsc_available,gsc_available_pct,avg_impressions_when_available,avg_position_when_available
0,0-30 days,1286535,1127183,87.61,166.99,11.38
1,Never optimized,8353753,2482061,29.71,37.23,19.07


### Staleness Verdict

**Verdict: CONFIRMED**

The staleness signal shows a clear and robust relationship with observed search performance.

Recently optimized content (0–30 days) has an average of 166.99 impressions and an average position of 11.38 among observations with GSC data available. Never-optimized content has substantially lower visibility, with 37.23 average impressions and an average position of 19.07.

The robustness check also shows a large difference in GSC availability between the groups: 87.61% for recently optimized observations versus 29.71% for never-optimized observations. Importantly, the relationship remains strong after restricting the performance comparison to observations where GSC data is available.

This supports using staleness as the primary prioritization signal for the baseline rule. The result is interpreted as an association rather than causal evidence: the analysis does not claim that optimization itself caused the difference in search performance.

The signal is based only on the content's recorded optimization date relative to the fixed March 2026 decision window and does not use future outcomes or label-derived inputs.

### Signal Audit Summary

| Signal | FlyRank linkage | Verdict | Baseline decision |
|---|---|---|---|
| Content staleness | Refresh logic | CONFIRMED | Use as primary scoring signal |
| Search volume | Quick-win / volume logic | MIXED | Do not use directly in score |

The audit supports a conservative baseline design. Staleness is the only confirmed scoring signal, while search volume is retained as contextual evidence rather than converted into a direct score because its observed relationship is non-monotonic.

The baseline therefore uses one primary signal, a transparent score, one reason code, and an explicit action label. GSC availability and impressions are treated as guardrails rather than additional scoring signals.

### Baseline Rule Design

The baseline uses content staleness as its single scoring signal because it was the only audited signal with a confirmed and robust relationship to observed search performance.

The rule prioritizes content that has never been optimized, while recently optimized content receives a lower priority.

GSC availability is used as a guardrail rather than as a scoring signal. Missing GSC data is not treated as evidence of poor performance.

Search volume is intentionally excluded from the score because its audited relationship was mixed and non-monotonic.

The rule produces exactly one reason code and one action label per observation, making the resulting queue deterministic, auditable, and easy to review.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Ranked Action Queue

The daily fact table contains repeated observations for the same client-content opportunity. For an operational action queue, the same opportunity should not occupy multiple ranked positions.

The queue therefore keeps the latest available March 2026 observation for each `client_hash_id × content_hash_id` pair. This preserves the baseline score and action while making the ranked list represent unique actionable opportunities.

The ranking remains deterministic: higher baseline score first, then higher observed impressions, then `content_hash_id` as a stable tie-breaker.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# BUILD UNIQUE RANKED ACTION QUEUE
# ============================================================

ranked_queue = con.execute(
    f"""
    WITH base AS (
        SELECT
            f.report_date,
            f.client_hash_id,
            f.content_hash_id,
            f.gsc_data_available,
            f.gsc_impressions,
            f.gsc_avg_position,
            c.last_optimized_date,

            ROW_NUMBER() OVER (
                PARTITION BY f.client_hash_id, f.content_hash_id
                ORDER BY f.report_date DESC
            ) AS rn

        FROM {MAR} f
        INNER JOIN read_parquet('{REL}/dim_content.parquet') c
            ON f.content_hash_id = c.content_hash_id

        WHERE c.is_deleted = FALSE
    ),

    latest AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            gsc_impressions,
            gsc_avg_position,
            last_optimized_date
        FROM base
        WHERE rn = 1
    )

    SELECT
        report_date,
        client_hash_id,
        content_hash_id,

        CASE
            WHEN last_optimized_date IS NULL THEN 100
            ELSE 10
        END AS action_score,

        CASE
            WHEN last_optimized_date IS NULL
                THEN 'NEVER_OPTIMIZED'
            ELSE 'RECENTLY_OPTIMIZED'
        END AS reason_code,

        CASE
            WHEN last_optimized_date IS NULL
                THEN 'REFRESH_HIGH_PRIORITY'
            ELSE 'MONITOR'
        END AS action,

        gsc_data_available,
        gsc_impressions,
        gsc_avg_position,
        last_optimized_date

    FROM latest

    ORDER BY
        action_score DESC,
        gsc_impressions DESC NULLS LAST,
        content_hash_id
    """
).df()

print(f"Unique ranked opportunities: {len(ranked_queue):,}")

ranked_queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Unique ranked opportunities: 324,947


,report_date,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_data_available,gsc_impressions,gsc_avg_position,last_optimized_date
0,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,14682,25.035826,NaT
1,2026-03-31,client_62f4a7e64f5e0096,content_f107e54b10b43725,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,8570,3.567561,NaT
2,2026-03-31,client_a80fca3f171ed1de,content_046fc480045b88f5,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,8492,7.605864,NaT
3,2026-03-31,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,7316,3.199973,NaT
4,2026-03-31,client_62f4a7e64f5e0096,content_acbcc847f8996314,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,6831,3.448104,NaT
5,2026-03-31,client_23a62021009f63c4,content_b05e73c8d4b617bb,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,6397,31.137721,NaT
6,2026-03-31,client_73cda7b4e4f265ea,content_f43118e089ecc69a,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,5845,5.462447,NaT
7,2026-03-31,client_23a62021009f63c4,content_73aa61dcedebbf30,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,5521,47.821953,NaT
8,2026-03-31,client_23a62021009f63c4,content_6530fa9d297c46eb,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,5364,89.848248,NaT
9,2026-03-31,client_20259bd6705d81d4,content_fa4b9e9229816684,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,True,5008,6.558706,NaT


In [12]:
# ============================================================
# QUEUE SANITY CHECK
# ============================================================

duplicate_pairs = ranked_queue.duplicated(
    subset=["client_hash_id", "content_hash_id"]
).sum()

print(f"Duplicate client-content pairs: {duplicate_pairs:,}")

print("\nAction distribution:")
print(
    ranked_queue["action"]
    .value_counts(dropna=False)
    .rename_axis("action")
    .reset_index(name="n")
)

print("\nReason-code distribution:")
print(
    ranked_queue["reason_code"]
    .value_counts(dropna=False)
    .rename_axis("reason_code")
    .reset_index(name="n")
)

Duplicate client-content pairs: 0

Action distribution:
                  action       n
0  REFRESH_HIGH_PRIORITY  282430
1                MONITOR   42517

Reason-code distribution:
          reason_code       n
0     NEVER_OPTIMIZED  282430
1  RECENTLY_OPTIMIZED   42517


### Write Baseline Action Queue

The final ranked queue is written from the notebook to the required output path.

The CSV is a reproducible artifact generated from the executed notebook and is intentionally kept out of git according to the repository's data-file policy.

In [14]:
# ============================================================
# WRITE FINAL BASELINE ACTION QUEUE
# ============================================================

from pathlib import Path

output_dir = Path("work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "baseline_action_score.csv"

ranked_queue.to_csv(
    output_path,
    index=False
)

print(f"Baseline action queue written to: {output_path}")
print(f"Rows written: {len(ranked_queue):,}")
print(f"Columns written: {len(ranked_queue.columns):,}")

Baseline action queue written to: work/outputs/baseline_action_score.csv
Rows written: 324,947
Columns written: 10


In [15]:
# ============================================================
# SECTION 2 — VERIFY THE WRITTEN CSV
# ============================================================

import os
import pandas as pd

assert os.path.exists(output_path), (
    f"CSV was not created: {output_path}"
)

written_queue = pd.read_csv(output_path)

# Row-count check
assert len(written_queue) == len(ranked_queue), (
    "CSV row count does not match the ranked queue."
)

# Required-column check
required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "action_score",
    "reason_code",
    "action",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_avg_position",
    "last_optimized_date",
]

missing_columns = [
    col for col in required_columns
    if col not in written_queue.columns
]

assert not missing_columns, (
    f"Missing required columns: {missing_columns}"
)

# Unique opportunity check
duplicate_pairs = written_queue[
    ["client_hash_id", "content_hash_id"]
].duplicated().sum()

assert duplicate_pairs == 0, (
    f"CSV contains {duplicate_pairs:,} duplicate client-content pairs."
)

# Reason-code and action checks
assert written_queue["reason_code"].notna().all(), (
    "Missing reason code detected."
)

assert written_queue["action"].notna().all(), (
    "Missing action detected."
)

# Score / reason / action consistency
assert (
    (
        (written_queue["action_score"] == 100)
        & (written_queue["reason_code"] == "NEVER_OPTIMIZED")
        & (written_queue["action"] == "REFRESH_HIGH_PRIORITY")
    )
    |
    (
        (written_queue["action_score"] == 10)
        & (written_queue["reason_code"] == "RECENTLY_OPTIMIZED")
        & (written_queue["action"] == "MONITOR")
    )
).all(), "Score, reason code, and action are inconsistent."

print("CSV verification: PASS")
print(f"Verified rows: {len(written_queue):,}")
print(
    "Verified unique client-content pairs: "
    f"{written_queue[['client_hash_id', 'content_hash_id']].drop_duplicates().shape[0]:,}"
)
print(f"Required columns present: {len(required_columns)}")
print(f"Duplicate client-content pairs: {duplicate_pairs:,}")

/tmp/ipykernel_23960/1191326664.py:12: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  written_queue = pd.read_csv(output_path)


CSV verification: PASS
Verified rows: 324,947
Verified unique client-content pairs: 324,947
Required columns present: 10
Duplicate client-content pairs: 0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-10 Review

The review below covers the ten highest-ranked unique client-content opportunities produced by the baseline queue.

For each opportunity, I record the assigned action, the reason it received that action, and the condition that would make the recommendation wrong.

This review is intentionally skeptical: a high baseline score indicates priority under the rule, not proof that a refresh will improve performance.

In [17]:
# ============================================================
# SECTION 3 — TOP-10 SKEPTICAL REVIEW
# ============================================================

top10_review = ranked_queue.head(10).copy()


def format_value(value, decimals=2):
    if pd.isna(value):
        return "NA"
    return f"{value:,.{decimals}f}"


def why_its_there(row):
    action = row["action"]
    reason = row["reason_code"]
    impressions = row["gsc_impressions"]
    position = row["gsc_avg_position"]

    evidence = (
        f"Observed impressions: {format_value(impressions, 0)}; "
        f"observed average position: {format_value(position, 2)}."
    )

    if reason == "NEVER_OPTIMIZED":
        return (
            f"{action} because the content has no recorded "
            f"last_optimized_date, which is the confirmed staleness "
            f"condition used by the baseline. {evidence}"
        )

    return (
        f"{action} because the content has a recorded optimization date "
        f"and therefore does not meet the baseline's never-optimized "
        f"condition. {evidence}"
    )


def what_would_make_it_wrong(row):
    if row["reason_code"] == "NEVER_OPTIMIZED":
        return (
            "The recommendation could be wrong if the missing "
            "last_optimized_date reflects incomplete metadata rather than "
            "true lack of optimization, or if the content is already "
            "performing well enough that a refresh is not warranted."
        )

    return (
        "The recommendation could be wrong if the recorded optimization "
        "date is inaccurate or if the content has deteriorated despite "
        "having been optimized."
    )


top10_review["why_its_there"] = top10_review.apply(
    why_its_there,
    axis=1
)

top10_review["what_would_make_it_wrong"] = top10_review.apply(
    what_would_make_it_wrong,
    axis=1
)


print(f"Top-10 rows reviewed: {len(top10_review)}")

top10_review[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_avg_position",
        "why_its_there",
        "what_would_make_it_wrong",
    ]
]

Top-10 rows reviewed: 10


,report_date,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_avg_position,why_its_there,what_would_make_it_wrong
0,2026-03-31,client_23a62021009f63c4,content_e6df0936699f5b8f,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,14682,25.035826,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
1,2026-03-31,client_62f4a7e64f5e0096,content_f107e54b10b43725,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,8570,3.567561,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
2,2026-03-31,client_a80fca3f171ed1de,content_046fc480045b88f5,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,8492,7.605864,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
3,2026-03-31,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,7316,3.199973,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
4,2026-03-31,client_62f4a7e64f5e0096,content_acbcc847f8996314,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,6831,3.448104,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
5,2026-03-31,client_23a62021009f63c4,content_b05e73c8d4b617bb,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,6397,31.137721,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
6,2026-03-31,client_73cda7b4e4f265ea,content_f43118e089ecc69a,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,5845,5.462447,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
7,2026-03-31,client_23a62021009f63c4,content_73aa61dcedebbf30,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,5521,47.821953,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
8,2026-03-31,client_23a62021009f63c4,content_6530fa9d297c46eb,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,5364,89.848248,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...
9,2026-03-31,client_20259bd6705d81d4,content_fa4b9e9229816684,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,5008,6.558706,REFRESH_HIGH_PRIORITY because the content has ...,The recommendation could be wrong if the missi...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks + Leakage Check

The baseline is intentionally simple, so some high-priority recommendations may be weak in an operational sense even when they satisfy the rule.

This section reviews questionable selections and checks the baseline inputs for future-window or label-derived leakage.

A weak pick is not treated as a failure of the baseline. Instead, it identifies where the simple rule could be wrong and where a future model may improve on it.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

# Review the highest-priority rows that may be operationally weak:
# they satisfy the staleness rule, but their observed performance
# suggests that a refresh may not necessarily be warranted.

weak_pick_mask = (
    (ranked_queue["reason_code"] == "NEVER_OPTIMIZED")
    & (ranked_queue["gsc_data_available"] == True)
    & (ranked_queue["gsc_avg_position"] <= 10)
)

weak_picks = ranked_queue.loc[
    weak_pick_mask,
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "action_score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_avg_position",
        "last_optimized_date",
    ]
].head(5).copy()

print(f"Weak/questionable picks reviewed: {len(weak_picks)}")

weak_picks


Weak/questionable picks reviewed: 5


,report_date,client_hash_id,content_hash_id,action_score,reason_code,action,gsc_impressions,gsc_avg_position,last_optimized_date
1,2026-03-31,client_62f4a7e64f5e0096,content_f107e54b10b43725,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,8570,3.567561,NaT
2,2026-03-31,client_a80fca3f171ed1de,content_046fc480045b88f5,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,8492,7.605864,NaT
3,2026-03-31,client_62f4a7e64f5e0096,content_f352b7cfd0b2f434,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,7316,3.199973,NaT
4,2026-03-31,client_62f4a7e64f5e0096,content_acbcc847f8996314,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,6831,3.448104,NaT
6,2026-03-31,client_73cda7b4e4f265ea,content_f43118e089ecc69a,100,NEVER_OPTIMIZED,REFRESH_HIGH_PRIORITY,5845,5.462447,NaT


In [19]:
# ============================================================
# SECTION 4 — LEAKAGE CHECK
# ============================================================

# Baseline inputs must be available at the decision moment.
# Explicitly document the fields used by the rule and verify that
# no outcome/label field or future-window aggregate is part of it.

baseline_input_columns = [
    "report_date",
    "last_optimized_date",
]

future_or_label_terms = [
    "label",
    "outcome",
    "future",
    "post",
    "conversion",
    "lift",
    "delta",
    "change",
]

suspected_leakage_columns = [
    col for col in ranked_queue.columns
    if any(term in col.lower() for term in future_or_label_terms)
]

print("Baseline rule inputs:")
for col in baseline_input_columns:
    print(f"  - {col}")

print("\nFuture/label-like columns present in ranked queue:")
print(
    suspected_leakage_columns
    if suspected_leakage_columns
    else "  None"
)

print("\nLeakage conclusion:")
print(
    "PASS — the baseline rule uses optimization staleness only "
    "and does not use future-window outcomes or label-derived inputs."
)

Baseline rule inputs:
  - report_date
  - last_optimized_date

Future/label-like columns present in ranked queue:
  None

Leakage conclusion:
PASS — the baseline rule uses optimization staleness only and does not use future-window outcomes or label-derived inputs.


### Weak-Pick Interpretation

These rows are intentionally treated as questionable rather than automatically wrong. They meet the confirmed staleness condition, but their observed search performance is already relatively strong, with average position at or above the first-page range used for this review.

A refresh could therefore be unnecessary or could introduce downside if the existing content is already satisfying search intent.

This is a useful limitation of the baseline: the rule prioritizes staleness, but it does not claim that every stale item will benefit from intervention.

### Leakage Conclusion

The baseline uses only decision-moment information available in the analysis slice. No future-window performance, post-action outcome, conversion result, or label-derived feature is used to assign the baseline score, reason code, or action.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.